# 05 - Field Extraction

## Purpose
Extracts structured field values from every classified document segment using
`AI_COMPLETE`. Extraction runs **once per parent file**: all pages of the file
are sent to the model a single time, together with the field schema for every
segment in that file. The model returns values for all segments in one
response, with provenance (source page and source label) for every value.

Because the model sees the whole shipment package, fields that are missing
from a segment's own pages can be resolved from other documents in the same
package, for example an HTS code on the Certificate of Origin, or a production
date and scientific name on a purchasing information sheet. Every value records
which page and which document it came from, so cross-document values are
visible and traceable.

## What this notebook does

**1. Load configuration**
- Extraction model loaded from `PIPELINE_CONFIG` (`extract_model`)
- Active field schemas loaded from `DOC_TYPE_CONFIG` for every doc type with
  segments not yet extracted by the active model
- One field instruction block per doc type is written to a temporary table,
  `FIELD_BLOCK_STAGING`, so the prompt can be assembled in SQL

**2. Build one prompt per parent file (SQL)**
- Each unextracted segment is labelled `S1`, `S2`, ... within its parent and
  listed with its doc type, page range, and its own field instructions.
  Short labels are used as JSON keys instead of UUIDs
- Every page of the parent is included once and tagged inline with the segment
  that owns it, e.g. `[PAGE 3 | S2 health_certificate]`. Pages belonging to
  `unknown` or already-extracted segments are tagged `not extracted: <type>`
  and still sent as context
- The package is wrapped in `<package>` tags, with the output instructions
  placed after the document. This prevents the model from continuing the
  document text instead of answering

**3. Source priority rules in the prompt**
- Extract each segment's fields from its own pages first
- Use other pages only when a field is genuinely absent from the segment's own
  pages, and only if that page refers to the same shipment (shared PO number,
  invoice number, container number, lot number, or product)
- Supporting documents (purchasing information sheets, lab reports, packing
  lists, letters of guarantee) are valid sources for product fields, even when
  issued by the buyer or a laboratory
- Party fields (supplier, exporter, facility, addresses) are only taken from
  another page if it names the same party in the same role. Slade Gorton is
  never used as a supplier, exporter, or facility

**4. Extract (parallel)**
- A single SQL statement calls `AI_COMPLETE` once per parent file;
  Snowflake parallelizes across files

**5. Parse and resolve provenance (Python)**
- Each response is parsed once per parent, then fanned out to its segments by
  `S` label
- For every field, `source_page` is compared with the segment's page range.
  If the value came from outside it, `IS_CROSS_DOC = TRUE`
- `source_page` is mapped back to the classified segment that owns it,
  recording `SOURCE_DOC_TYPE` and `SOURCE_CHILD_DOC_ID`
- `SOURCE_DOC_LABEL` is a display version of the source doc type. For
  `unknown` or unassigned pages it adds a short title in parentheses, taken
  from the page's own heading (falling back to the segment's title or
  description), e.g.
  `unknown (SLADE GORTON AND CO. INC PURCHASING INFORMATION SHEET)`

**6. Write and update status**
- Results written to `DOCUMENTS_EXTRACTED` and `DOCUMENTS_EXTRACTED_FLAT`
- Token estimates written to `LLM_USAGE`, one row per parent call
- `DOCUMENTS_INGESTED.STATUS` updated to `EXTRACTED` or `EXTRACT_ERROR`

## Outputs
| Table | Grain | What is written |
|---|---|---|
| `PROCESSING.DOCUMENTS_EXTRACTED` | One row per segment | That segment's extraction JSON, doc type, model |
| `PROCESSING.DOCUMENTS_EXTRACTED_FLAT` | One row per field per segment | Value, confidence, mandatory flag, missing flag, `SOURCE_PAGE`, `SOURCE_LABEL`, `IS_CROSS_DOC`, `SOURCE_DOC_TYPE`, `SOURCE_CHILD_DOC_ID`, `SOURCE_DOC_LABEL` |
| `AUDIT.LLM_USAGE` | One row per parent file | Estimated input and output tokens for the extraction call |
| `INGEST.DOCUMENTS_INGESTED` | One row per file | STATUS updated to `EXTRACTED` or `EXTRACT_ERROR` |


## Key design decisions
- **One call per parent file** - pages are sent once regardless of how many
  segments a file contains. Single-document files (`CHILD_DOC_ID = DOC_ID`)
  behave exactly as a per-segment extraction would
- **Provenance on every field** - `SOURCE_PAGE` and `SOURCE_LABEL` make errors
  debuggable (e.g. whether a supplier name was read from the letterhead or the
  SHIPPER block) and make cross-document resolution auditable
- **Config-driven** - the model comes from `PIPELINE_CONFIG` and field
  questions from `DOC_TYPE_CONFIG`; no notebook edits are needed to change
  either

In [ ]:
import json
import re
import pandas as pd
from snowflake.snowpark.context import get_active_session

DB                = 'PERMAFROST_POC'
INGEST_SCHEMA     = 'INGEST'
PROCESSING_SCHEMA = 'PROCESSING'
CONFIG_SCHEMA     = 'CONFIG'
AUDIT_SCHEMA      = 'AUDIT'

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

def estimate_tokens(text):
    return len(text) // 4 if text else 0

def is_empty(value):
    """
    Returns True if value is absent, None, empty string,
    the literal string 'None', an empty list, or an empty dict.
    """
    if value is None:
        return True
    if isinstance(value, str) and value.strip().lower() in ('none', ''):
        return True
    if isinstance(value, (list, dict)) and len(value) == 0:
        return True
    return False

def parse_ai_response(raw):
    if raw is None:
        raise ValueError("NULL response from AI_COMPLETE")
    stripped = raw.strip()
    if stripped.startswith('"') and stripped.endswith('"'):
        stripped = json.loads(stripped)
    if '```' in stripped:
        parts   = stripped.split('```')
        content = parts[1]
        if content.startswith('json'):
            content = content[4:]
        stripped = content.strip()
    result = json.loads(stripped)
    if isinstance(result, str):
        result = json.loads(result)
    if not isinstance(result, dict):
        raise ValueError(f"Expected dict, got {type(result).__name__}")
    return result

TITLE_KEYWORDS = (
    'CERTIFICATE', 'INVOICE', 'PACKING', 'LIST', 'BILL OF LADING', 'WAYBILL',
    'REPORT', 'SHEET', 'DECLARATION', 'LETTER', 'GUARANTEE', 'ATTACHMENT',
    'STATEMENT', 'FORM', 'AFFIDAVIT', 'ANALYSIS', 'WORKSHEET',
)

def page_title(head_text, max_len=70):
    """Best-guess document title from the first lines of a page."""
    if not head_text:
        return None
    candidates = []
    for ln in head_text.splitlines()[:15]:
        t = re.sub(r'[#*|>`]', ' ', ln)          # strip markdown
        t = re.sub(r'\s+', ' ', t).strip(' -:')
        if len(re.findall(r'[A-Za-z]', t)) >= 3:  # skip blank, CJK-only, table separators
            candidates.append(t)
    if not candidates:
        return None
    for t in candidates:                          # prefer a line that names a document type
        if any(k in t.upper() for k in TITLE_KEYWORDS):
            return t[:max_len]
    return candidates[0][:max_len]


def resolve_source(page_map, page_heads, source_page):
    """
    Returns (doc_type, child_doc_id, display_label) for the page a value came from.
    For unknown or unassigned pages, the label adds a short title in parentheses.
    """
    if source_page is None:
        return None, None, None

    seg = next((s for s in page_map
                if s['page_start'] <= source_page <= s['page_end']), None)
    doc_type = seg['doc_type'] if seg else 'unassigned'
    child_id = seg['child_doc_id'] if seg else None
    label    = doc_type

    if doc_type in ('unknown', 'unassigned'):
        title = page_title(page_heads.get(str(source_page)))
        if not title and seg and seg.get('description'):
            title = seg['description'][:70]
        if title:
            label = f"{doc_type} ({title})"

    return doc_type, child_id, label

# Date normalization and item reference helpers
from datetime import datetime

DATE_FIELDS = {"production_date", "document_date", "expiry_date", "catch_date"}

DMY_FORMATS = [
    "%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y", "%d-%b-%Y", "%d-%b-%y",
    "%d %B %Y", "%d %b %Y", "%B %d, %Y", "%b %d, %Y", "%b. %d, %Y",
    "%b-%d-%Y", "%d/%m/%y", "%Y/%m/%d", "%Y.%m.%d",
]
MDY_FORMATS = [
    "%Y-%m-%d", "%m/%d/%Y", "%m/%d/%y", "%d-%b-%Y", "%d-%b-%y",
    "%B %d, %Y", "%b %d, %Y", "%b-%d-%Y",
]
RANGE_SPLIT = re.compile(r"\s+(?:to|-|~|–)\s+")
ITEM_PREFIX = re.compile(r"^\s*(\[[^\]]+\])\s*(.*)$")

def split_item_ref(s):
    """'[2,4] 2022-04-03' -> ('[2,4]', '2022-04-03'); untagged -> (None, s)."""
    m = ITEM_PREFIX.match(s) if isinstance(s, str) else None
    return (m.group(1), m.group(2)) if m else (None, s)

def expand_item_ref(ref):
    """'[3,6,7]' -> {3,6,7}; '[1-7]' -> {1..7}; None -> empty set."""
    items = set()
    if not ref:
        return items
    for part in ref.strip("[]").split(","):
        part = part.strip()
        if "-" in part:
            a, b = part.split("-", 1)
            if a.strip().isdigit() and b.strip().isdigit():
                items.update(range(int(a), int(b) + 1))
        elif part.isdigit():
            items.add(int(part))
    return items

def normalize_date(raw, month_first=False):
    """One date - YYYY-MM-DD. Returns the input unchanged if no format matches."""
    if not isinstance(raw, str):
        return raw
    s_ = raw.strip().rstrip(".")
    for fmt in (MDY_FORMATS if month_first else DMY_FORMATS):
        try:
            return datetime.strptime(s_, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return raw

def normalize_date_or_range(raw, month_first=False):
    parts = RANGE_SPLIT.split(raw.strip()) if isinstance(raw, str) else [raw]
    if len(parts) == 2:
        return f"{normalize_date(parts[0], month_first)} to {normalize_date(parts[1], month_first)}"
    return normalize_date(raw, month_first)

def normalize_date_field(value, month_first=False):
    """String, list, dict {'date': ...}, or list of dicts -> None, string, or list of strings.
    Keeps [n] prefixes; dedups only exact duplicates (same prefix and same date)."""
    if value is None:
        return None
    out = []
    for item in (value if isinstance(value, list) else [value]):
        if isinstance(item, dict):
            item = item.get("date") or item.get("value")
        if not (isinstance(item, str) and item.strip()):
            continue
        ref, body = split_item_ref(item)
        norm = normalize_date_or_range(body, month_first)
        tagged = f"{ref} {norm}" if ref else norm
        if tagged not in out:
            out.append(tagged)
    if not out:
        return None
    return out[0] if len(out) == 1 else out

def collect_item_refs(value):
    """All item numbers referenced by a (possibly tagged) field value."""
    refs = set()
    for v in (value if isinstance(value, list) else [value]):
        ref, _ = split_item_ref(v)
        refs |= expand_item_ref(ref)
    return refs

def find_item_mismatch(item_refs):
    """item_refs: {scope: {field_id: set(items)}}. Flags item numbers that only
    one field references, e.g. catch_date tagged [8] with no vessel for item 8."""
    issues = []
    for scope, by_field in item_refs.items():
        if len(by_field) < 2:
            continue
        for fid, refs in by_field.items():
            others = set().union(*(r for f, r in by_field.items() if f != fid))
            orphans = refs - others
            if orphans:
                issues.append(f"{fid}:{sorted(orphans)}")
    return issues

def is_month_first(seg_result):
    """US-issued certificates (NOAA/NMFS) write dates MM/DD/YYYY."""
    va = seg_result.get("validating_authority")
    va = va.get("value") if isinstance(va, dict) else va
    return "NATIONAL MARINE FISHERIES" in str(va or "").upper()

s = get_active_session()

In [ ]:
# Load EXTRACT_MODEL from PIPELINE_CONFIG 
config = {
    row['CONFIG_KEY']: row['CONFIG_VALUE']
    for row in s.sql(f"""
        SELECT CONFIG_KEY, CONFIG_VALUE
        FROM {DB}.{CONFIG_SCHEMA}.PIPELINE_CONFIG
        WHERE CONFIG_KEY = 'extract_model'
          AND IS_ACTIVE  = TRUE
    """).collect()
}

EXTRACT_MODEL = config.get('extract_model')

if not EXTRACT_MODEL:
    raise ValueError(
        "Missing 'extract_model' in PIPELINE_CONFIG — "
        "run 00_setup_config.ipynb first"
    )

info(f"Extract model: {EXTRACT_MODEL}")

In [ ]:
# one call covers every segment in a parent file
PACKAGE_PROMPT_TEMPLATE = """You are extracting structured data from a shipment document package for a seafood importer (Slade Gorton).

The package contains one or more document segments. Each segment has a SEGMENT label, a document type, a page range, and its own list of fields to extract.

SEGMENTS AND FIELDS:
{segments_block}

SOURCE PRIORITY - apply to every field of every segment:
0. Each page is tagged with the segment it belongs to, e.g. [PAGE 3 | S2 health_certificate]. A segment's own pages are the pages tagged with its label.
1. Extract each segment's fields from that segment's own pages first.
2. If a field is genuinely absent from the segment's own pages, look for it on the other pages of the package, including pages tagged 'not extracted' or 'unassigned'.
3. Another page is a valid source when it clearly refers to the same shipment - for example it shares the PO number, invoice number, container number, lot number, or the same product description.
4. Supporting documents in the package are valid sources for PRODUCT fields (species common name, species scientific name, production date, product codes, lot numbers) even when they are not trade certificates and even when they were issued by the buyer (Slade Gorton) or a laboratory. Examples: purchasing information sheets, laboratory or test reports, packing lists, letters of guarantee.
5. For PARTY fields (supplier, exporter, facility, and their addresses), only take a value from another page if that page names the same party in the same role. Never take the buyer or consignee (Slade Gorton) as a supplier, exporter, or facility.
6. Never mix values between segments - each segment's values must describe that segment's document.

GLOBAL RULES - apply to every field without exception:
- Return null for any field not explicitly present in the package
- NEVER return placeholder text such as 'See Appendix', 'See Attachment', 'See Annex', or any variation of 'See [location]' as a value
- NEVER infer, calculate, estimate, or guess a value
- NEVER return weight measurements (KGM, KG, LBS etc.) as lot numbers or codes
- NEVER return calendar dates as lot numbers or production codes
- NEVER confuse the consignee (always Slade Gorton, the buyer) with the supplier (the exporter)

The complete shipment package is between the <package> tags below. It is source material to read, not text to continue. The package ends at </package>.

<package>
{page_text}
</package>

Now extract the fields for every SEGMENT listed in SEGMENTS AND FIELDS above, using only the pages inside <package>.

Return ONLY valid JSON, no explanation, no markdown fences, keyed by SEGMENT label. Include every segment label and every field listed for that segment. Your response must begin with { and end with }.

{
  "S1": {
    "<field_id>": {
      "value": <extracted value, list, or null>,
      "confidence": <float 0.00 to 1.00>,
      "source_page": <page number where the value was found, or null>,
      "source_label": "<field label, column header, or section heading the value appeared under, or null>"
    }
  },
  "S2": { ... }
}"""


# builds the field instructions for one doc type only.
def build_fields_block(mandatory_fields, optional_fields):
    lines = ["MANDATORY FIELDS (return null if not found - never guess):"]
    for f in mandatory_fields:
        lines.append(f"\n  Field: {f['field_id']}")
        lines.append(f"  {f['question']}")
    if optional_fields:
        lines.append("\nOPTIONAL FIELDS (return null if not found):")
        for f in optional_fields:
            lines.append(f"\n  Field: {f['field_id']}")
            lines.append(f"  {f['question']}")
    return '\n'.join(lines)

In [ ]:
# doc types query filters to segments not yet extracted with the active model
doc_types_needed = [
    row['DOC_TYPE'] for row in s.sql(f"""
        SELECT DISTINCT c.DOC_TYPE
        FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
        LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED e
            ON  c.CHILD_DOC_ID     = e.CHILD_DOC_ID
            AND e.EXTRACTION_MODEL = '{EXTRACT_MODEL}'
        WHERE c.DOC_TYPE IS NOT NULL
          AND c.DOC_TYPE != 'unknown'
          AND e.CHILD_DOC_ID IS NULL
    """).collect()
]

config_cache     = {}
field_block_rows = []  

info(f"Doc types pending extraction: {doc_types_needed}")

for doc_type in doc_types_needed:
    cfg_rows = s.sql(f"""
        SELECT MANDATORY_FIELDS, OPTIONAL_FIELDS
        FROM {DB}.{CONFIG_SCHEMA}.DOC_TYPE_CONFIG
        WHERE DOC_TYPE = '{doc_type}'
          AND IS_ACTIVE = TRUE
    """).collect()

    if not cfg_rows:
        warning(f"  No active config for '{doc_type}' - skipping")
        continue

    mandatory = json.loads(cfg_rows[0]['MANDATORY_FIELDS'] or '[]')
    optional  = json.loads(cfg_rows[0]['OPTIONAL_FIELDS']  or '[]')

    if not mandatory:
        warning(f"  No mandatory fields for '{doc_type}' - skipping")
        continue

    all_fields = mandatory + optional
    config_cache[doc_type] = {
        'mandatory_ids': {f['field_id'] for f in mandatory},
        'all_field_ids': {f['field_id'] for f in all_fields},
        'item_scope':    {f['field_id']: f.get('item_ref_scope')
                          for f in all_fields if f.get('item_tagged')},
    }

    # field block staged for SQL-side prompt assembly
    field_block_rows.append({
        'DOC_TYPE':     doc_type,
        'FIELDS_BLOCK': build_fields_block(mandatory, optional),
    })

    info(f"  [{doc_type}] {len(mandatory)} mandatory + {len(optional)} optional fields")

if field_block_rows:
    s.write_pandas(
        pd.DataFrame(field_block_rows),
        table_name='FIELD_BLOCK_STAGING',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=True,
        auto_create_table=True,
        table_type='temporary',
    )
    info(f"Staged field blocks for {len(field_block_rows)} doc type(s)")
else:
    print("\nNo valid configs found - nothing to extract.")

In [ ]:
if config_cache:
    all_extracted_rows = []
    all_flat_rows      = []
    all_llm_rows       = []
    all_errors         = []
    parent_calls       = 0
    cross_doc_total    = 0

    DOCUMENTS_CLASSIFIED = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED"
    DOCUMENTS_PAGES      = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES"
    DOCUMENTS_EXTRACTED  = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED"
    DOCUMENTS_INGESTED   = f"{DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED"
    FIELD_BLOCKS         = f"{DB}.{PROCESSING_SCHEMA}.FIELD_BLOCK_STAGING"

    def update_parent_status(child_doc_ids, status):
        if not child_doc_ids:
            return
        escaped = [str(i).replace("'", "''") for i in child_doc_ids]
        id_list = ', '.join(f"'{i}'" for i in escaped)
        s.sql(f"""
            UPDATE {DOCUMENTS_INGESTED}
            SET STATUS = '{status}'
            WHERE DOC_ID IN (
                SELECT DOC_ID
                FROM {DOCUMENTS_CLASSIFIED}
                WHERE CHILD_DOC_ID IN ({id_list})
            )
        """).collect()

    info("Extracting pending segments - one AI_COMPLETE call per parent file ...")

    # Snowflake parallelizes AI_COMPLETE across parent files.
    try:
        results = s.sql(f"""
            WITH segs AS (
                -- Unextracted segments joined to their doc type's field block,
                -- labelled S1, S2, ... within each parent file
                SELECT
                    c.DOC_ID,
                    c.CHILD_DOC_ID,
                    c.DOC_TYPE,
                    c.PAGE_START,
                    c.PAGE_END,
                    fb.FIELDS_BLOCK,
                    'S' || ROW_NUMBER() OVER (
                        PARTITION BY c.DOC_ID
                        ORDER BY c.PAGE_START, c.CHILD_DOC_ID
                    )                                              AS SEG_LABEL
                FROM {DOCUMENTS_CLASSIFIED} c
                JOIN {FIELD_BLOCKS} fb
                    ON c.DOC_TYPE = fb.DOC_TYPE
                LEFT JOIN {DOCUMENTS_EXTRACTED} e
                    ON  c.CHILD_DOC_ID     = e.CHILD_DOC_ID
                    AND e.EXTRACTION_MODEL = '{EXTRACT_MODEL}'
                WHERE e.CHILD_DOC_ID IS NULL
            ),
            seg_block AS (
                -- One segments block + segment metadata per parent
                SELECT
                    DOC_ID,
                    COUNT(*)                                       AS SEGMENT_COUNT,
                    LISTAGG(
                        '=== SEGMENT ' || SEG_LABEL || ' | ' || DOC_TYPE ||
                        ' | pages ' || PAGE_START || '-' || PAGE_END || ' ===' ||
                        CHR(10) || FIELDS_BLOCK,
                        '\\n\\n'
                    ) WITHIN GROUP (ORDER BY PAGE_START)           AS SEGMENTS_BLOCK,
                    ARRAY_AGG(OBJECT_CONSTRUCT(
                        'seg_label',    SEG_LABEL,
                        'child_doc_id', CHILD_DOC_ID,
                        'doc_type',     DOC_TYPE,
                        'page_start',   PAGE_START,
                        'page_end',     PAGE_END
                    ))                                             AS SEGMENTS_META
                FROM segs
                GROUP BY DOC_ID
            ),
            parent_map AS (
                -- every classified segment in the parent (including unknown
                -- and already-extracted), used to resolve source_page -> doc type
                SELECT
                    DOC_ID,
                    ARRAY_AGG(OBJECT_CONSTRUCT(
                        'child_doc_id', CHILD_DOC_ID,
                        'doc_type',     DOC_TYPE,
                        'page_start',   PAGE_START,
                        'page_end',     PAGE_END,
                        'description',  DOCUMENT_DESCRIPTION
                    ))                                             AS PAGE_MAP
                FROM {DOCUMENTS_CLASSIFIED}
                WHERE DOC_ID IN (SELECT DOC_ID FROM seg_block)
                GROUP BY DOC_ID
            ),
            page_heads AS (
               -- opening text of each page, for deriving a document title
                SELECT
                    DOC_ID,
                    OBJECT_AGG(
                        PAGE_NUMBER::VARCHAR,
                        LEFT(COALESCE(PAGE_CONTENT_TRANSLATED, PAGE_CONTENT), 600)::VARIANT
                    )                                                  AS PAGE_HEADS
                FROM {DOCUMENTS_PAGES}
                WHERE DOC_ID IN (SELECT DOC_ID FROM seg_block)
                  AND PAGE_CONTENT IS NOT NULL
                GROUP BY DOC_ID
            ),
            page_tags AS (
                -- One tag per page - which segment owns it
                SELECT
                    p.DOC_ID,
                    p.PAGE_NUMBER,
                    COALESCE(
                        MAX(sg.SEG_LABEL || ' ' || sg.DOC_TYPE),
                        'not extracted: ' || MAX(ac.DOC_TYPE),
                        'unassigned'
                    )                                              AS PAGE_TAG
                FROM {DOCUMENTS_PAGES} p
                LEFT JOIN segs sg
                    ON  p.DOC_ID = sg.DOC_ID
                    AND p.PAGE_NUMBER BETWEEN sg.PAGE_START AND sg.PAGE_END
                LEFT JOIN {DOCUMENTS_CLASSIFIED} ac
                    ON  p.DOC_ID = ac.DOC_ID
                    AND p.PAGE_NUMBER BETWEEN ac.PAGE_START AND ac.PAGE_END
                WHERE p.DOC_ID IN (SELECT DOC_ID FROM seg_block)
                GROUP BY p.DOC_ID, p.PAGE_NUMBER
            ),
            page_block AS (
                -- All pages of the parent sent once, each tagged inline
                SELECT
                    p.DOC_ID,
                    COUNT(*)                                       AS PAGE_COUNT,
                    SUM(LENGTH(COALESCE(
                        p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT
                    )))                                            AS TOTAL_CHARS,
                    LISTAGG(
                        '[PAGE ' || p.PAGE_NUMBER || ' | ' || t.PAGE_TAG || ']' ||
                        CHR(10) ||
                        COALESCE(p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT),
                        '\\n\\n'
                    ) WITHIN GROUP (ORDER BY p.PAGE_NUMBER)        AS PAGE_TEXT
                FROM {DOCUMENTS_PAGES} p
                JOIN page_tags t
                    ON  p.DOC_ID      = t.DOC_ID
                    AND p.PAGE_NUMBER = t.PAGE_NUMBER
                WHERE p.PAGE_CONTENT IS NOT NULL
                GROUP BY p.DOC_ID
            )
            SELECT
                sb.DOC_ID,
                sb.SEGMENT_COUNT,
                sb.SEGMENTS_META,
                pm.PAGE_MAP,     
                ph.PAGE_HEADS,
                pb.PAGE_COUNT,
                pb.TOTAL_CHARS,
                LENGTH(sb.SEGMENTS_BLOCK)                          AS SEGMENTS_CHARS,
                AI_COMPLETE(
                    '{EXTRACT_MODEL}',
                    REPLACE(
                        REPLACE(?, '{{segments_block}}', sb.SEGMENTS_BLOCK),
                        '{{page_text}}', pb.PAGE_TEXT
                    ),
                    {{'max_tokens': 16000}}
                )                                                  AS EXTRACTED
            FROM seg_block sb
            JOIN page_block pb
                ON sb.DOC_ID = pb.DOC_ID
            JOIN parent_map pm                          
                ON sb.DOC_ID = pm.DOC_ID
            JOIN page_heads ph 
                ON sb.DOC_ID = ph.DOC_ID 
            ORDER BY sb.DOC_ID
        """, params=[PACKAGE_PROMPT_TEMPLATE]).collect()

        info(f"  {len(results)} parent file(s) returned from Cortex")

    except Exception as exc:
        error(f"  SQL extraction failed: {exc}")
        results = []

    # Parse per parent, then fan out to segments
    for row in results:
        doc_id   = row['DOC_ID']
        segments = json.loads(row['SEGMENTS_META'])
        page_map = json.loads(row['PAGE_MAP'])        
        page_heads = json.loads(row['PAGE_HEADS'] or '{}') 
        raw      = row['EXTRACTED']
        parent_calls += 1

        # One LLM_USAGE row per parent call
        all_llm_rows.append({
            'DOC_ID':        doc_id,
            'CHILD_DOC_ID':  None,
            'PIPELINE_STEP': 'EXTRACTION',
            'MODEL_NAME':         EXTRACT_MODEL,
            'TOKENS_IN':     (len(PACKAGE_PROMPT_TEMPLATE)
                              + (row['SEGMENTS_CHARS'] or 0)
                              + (row['TOTAL_CHARS'] or 0)) // 4,
            'TOKENS_OUT':    estimate_tokens(raw),
        })

        # Parent-level parse - failure marks every segment in this parent
        try:
            result = parse_ai_response(raw)
        except Exception as exc:
            for seg in segments:
                all_errors.append({
                    'child_doc_id': seg['child_doc_id'],
                    'doc_type':     seg['doc_type'],
                    'error':        f"Parent-level parse failure: {exc}",
                })
            error(f"  [FAIL] parent {doc_id} ({len(segments)} segment(s)): {exc}")
            continue

        expected_labels = {seg['seg_label'] for seg in segments}
        missing_labels  = expected_labels - set(result.keys())
        if missing_labels:
            warning(f"  Parent {doc_id}: response missing segment(s) {sorted(missing_labels)}")

        info(f"  Parent {doc_id} - {row['SEGMENT_COUNT']} segment(s), "
             f"{row['PAGE_COUNT']} page(s)")

        for seg in segments:
            label        = seg['seg_label']
            child_doc_id = seg['child_doc_id']
            doc_type     = seg['doc_type']
            page_start   = seg['page_start']
            page_end     = seg['page_end']
            cfg          = config_cache[doc_type]

            seg_result = result.get(label)
            if not isinstance(seg_result, dict):
                all_errors.append({
                    'child_doc_id': child_doc_id,
                    'doc_type':     doc_type,
                    'error':        f"Segment {label} missing from response",
                })
                error(f"    [FAIL] {child_doc_id} ({label}): not in model response")
                continue

            all_extracted_rows.append({
                'CHILD_DOC_ID':     child_doc_id,
                'DOC_TYPE':         doc_type,
                'EXTRACTED_JSON':   json.dumps(seg_result),
                'EXTRACTION_MODEL': EXTRACT_MODEL,
            })
            
            month_first      = is_month_first(seg_result)
            item_refs        = {}
            cross_doc_fields = []
            missing_mand     = []   # mandatory fields missing after normalization

            for field_id in cfg['all_field_ids']:
                field_data = seg_result.get(field_id, {})

                # Read provenance alongside value and confidence
                if isinstance(field_data, dict):
                    value        = field_data.get('value')
                    confidence   = field_data.get('confidence')
                    source_page  = field_data.get('source_page')
                    source_label = field_data.get('source_label')
                else:
                    value, confidence, source_page, source_label = field_data, None, None, None

                # Normalise source_page to int
                try:
                    source_page = int(source_page) if source_page is not None else None
                except (TypeError, ValueError):
                    source_page = None

                # Normalize dates to YYYY-MM-DD, keeping any [n] item prefixes
                if field_id in DATE_FIELDS:
                    value = normalize_date_field(value, month_first)

                # Collect item numbers for the alignment check
                scope = cfg['item_scope'].get(field_id)
                if scope and not is_empty(value):
                    refs = collect_item_refs(value)
                    if refs:
                        item_refs.setdefault(scope, {})[field_id] = refs

                missing = is_empty(value)
                if missing and field_id in cfg['mandatory_ids']:
                    missing_mand.append(field_id)

                # Cross-doc if the value came from outside this segment's pages
                is_cross_doc = (
                    not missing
                    and source_page is not None
                    and not (page_start <= source_page <= page_end)
                )

                # which document the value came from
                _, src_child_id, src_doc_label = (
                    resolve_source(page_map, page_heads, source_page) if not missing else (None, None, None)
                )

                if is_cross_doc:
                    cross_doc_fields.append(f"{field_id}@{src_doc_label}:p{source_page}")

                all_flat_rows.append({
                    'CHILD_DOC_ID':        child_doc_id,
                    'FIELD_ID':            field_id,
                    'FIELD_VALUE':         None if missing else str(value),
                    'FIELD_CONFIDENCE':    confidence,
                    'IS_MANDATORY':        field_id in cfg['mandatory_ids'],
                    'IS_MISSING':          missing,
                    'SOURCE_PAGE':         None if missing else source_page,
                    'SOURCE_LABEL':        None if missing else source_label,
                    'IS_CROSS_DOC':        is_cross_doc,
                    'SOURCE_CHILD_DOC_ID': src_child_id,
                    'SOURCE_DOC_LABEL':    src_doc_label,
                    'EXTRACTION_MODEL':    EXTRACT_MODEL,
                })

            cross_doc_total += len(cross_doc_fields)

            # Flag item numbers referenced by only one field
            item_mismatch = find_item_mismatch(item_refs)
            if item_mismatch:
                warning(f"    ITEM_REF_MISMATCH {child_doc_id} ({label}): {item_mismatch}")

            # Log includes segment label, page range, cross-doc fields with source doc type
            msg = f"    [OK] {child_doc_id} ({label} {doc_type} pp.{page_start}-{page_end})"
            if missing_mand:
                msg += f" | missing: {sorted(missing_mand)}"
            if cross_doc_fields:
                msg += f" | cross-doc: {cross_doc_fields}"
            info(msg)

In [ ]:
# Write DOCUMENTS_EXTRACTED

if all_extracted_rows:
    s.write_pandas(
        pd.DataFrame(all_extracted_rows),
        table_name="DOCUMENTS_EXTRACTED",
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"\nWrote {len(all_extracted_rows)} row(s) to DOCUMENTS_EXTRACTED")

#  Write DOCUMENTS_EXTRACTED_FLAT

if all_flat_rows:
    s.write_pandas(
        pd.DataFrame(all_flat_rows),
        table_name="DOCUMENTS_EXTRACTED_FLAT",
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(all_flat_rows)} row(s) to DOCUMENTS_EXTRACTED_FLAT")

In [ ]:
# Write LLM_USAGE 

if all_llm_rows:
    s.write_pandas(
        pd.DataFrame(all_llm_rows),
        table_name="LLM_USAGE",
        database=DB, schema=AUDIT_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(all_llm_rows)} row(s) to LLM_USAGE")
# Update STATUS in DOCUMENTS_INGESTED

extracted_ids = [row["CHILD_DOC_ID"] for row in all_extracted_rows]
error_ids     = [row["CHILD_DOC_ID"] for row in all_errors]

if extracted_ids:
    update_parent_status(extracted_ids, "EXTRACTED")
    info(f"Updated {len(extracted_ids)} document(s) to EXTRACTED")

if error_ids:
    update_parent_status(error_ids, "EXTRACT_ERROR")
    info(f"Updated {len(error_ids)} document(s) to EXTRACT_ERROR")

In [ ]:
# Summary

print(f"\n Extraction summary")
print(f"  Extracted successfully : {len(all_extracted_rows)}")
print(f"  Fields extracted       : {len(all_flat_rows)}")
print(f"  Errors                 : {len(all_errors)}")
print(f"  Total input tokens     : {sum(r['TOKENS_IN'] for r in all_llm_rows):,}")
print(f"  Total output tokens    : {sum(r['TOKENS_OUT'] for r in all_llm_rows):,}")

if all_errors:
    print("\n  Failed documents:")
    for e in all_errors:
        print(f"    {e['child_doc_id']} ({e['doc_type']}): {e['error']}")

print(f"\n Results by doc type")
s.sql(f"""
        SELECT
            c.DOC_TYPE,
            COUNT(DISTINCT e.CHILD_DOC_ID)              AS DOCS_EXTRACTED,
            COUNT(*)                                     AS TOTAL_FIELDS,
            COUNT(CASE WHEN e.IS_MISSING = TRUE
                       AND e.IS_MANDATORY THEN 1 END)   AS MISSING_MANDATORY,
            ROUND(AVG(e.FIELD_CONFIDENCE), 3)            AS AVG_CONFIDENCE
        FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED_FLAT e
        JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
            ON e.CHILD_DOC_ID = c.CHILD_DOC_ID
        GROUP BY c.DOC_TYPE
        ORDER BY c.DOC_TYPE
""").show()